In [10]:
import os
import json
import chromadb
import mlflow
import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import Annotated, TypedDict
from langchain_core.messages import HumanMessage, AIMessage

# load environment
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), '.env'))
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# initialize LLM and embedder
# using a more capable model for agent tool calling
llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY, temperature=0.3)
embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

# connect to existing ChromaDB - already built in RAG notebook
chroma_client = chromadb.PersistentClient(path="../data/processed/chromadb")
comments_collection = chroma_client.get_collection("comments")
transcripts_collection = chroma_client.get_collection("transcripts")
summaries_collection = chroma_client.get_collection("topic_summaries")

# MLflow setup
mlflow.set_tracking_uri("sqlite:///../mlruns/mlflow.db")
mlflow.set_experiment("youtube-intelligence-engine-agent")
mlflow.langchain.autolog()

print("Collections loaded:")
print(f"  comments: {comments_collection.count()}")
print(f"  transcripts: {transcripts_collection.count()}")
print(f"  summaries: {summaries_collection.count()}")
print("LLM ready:", llm.model_name)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14175.53it/s]


Collections loaded:
  comments: 27330
  transcripts: 261
  summaries: 15
LLM ready: llama-3.3-70b-versatile


In [11]:
# ── core retrieval function ──
# reused from RAG notebook - semantic search on any collection
def semantic_retrieval(query, collection, k=5, filter_dict=None):
    query_embedding = embedder.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        where=filter_dict
    )
    return results["documents"][0], results["metadatas"][0]


# ── hybrid retrieval across all three collections ──
def hybrid_retrieval(query, k_each=3):
    comment_docs, comment_meta = semantic_retrieval(query, comments_collection, k=k_each)
    transcript_docs, transcript_meta = semantic_retrieval(query, transcripts_collection, k=k_each)
    summary_docs, summary_meta = semantic_retrieval(query, summaries_collection, k=2)
    return {
        "comments":   list(zip(comment_docs, comment_meta)),
        "transcripts": list(zip(transcript_docs, transcript_meta)),
        "summaries":  list(zip(summary_docs, summary_meta))
    }


# ── format hybrid results into one context string for LLM ──
def format_context(hybrid_results):
    comment_context = "\n\n".join([doc for doc, _ in hybrid_results["comments"]])
    transcript_context = "\n\n".join([doc for doc, _ in hybrid_results["transcripts"]])
    summary_context = "\n\n".join([doc for doc, _ in hybrid_results["summaries"]])
    return comment_context, transcript_context, summary_context


print("Retrieval functions defined.")

Retrieval functions defined.


In [20]:
# ── QA prompt for final answer generation ──
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an AI analyst specializing in public opinion about AI and the future of work.
Answer questions based on real YouTube comments and video transcripts.
Always ground your answers in the provided context.
Never make up opinions not present in the context."""),
    ("human", "{input}")
])

# ── TOOL 1: General RAG Q&A ──
@tool
def rag_qa(question: str) -> str:
    """Answer general questions about AI and the future of work using 
    YouTube comments, transcripts, and topic summaries."""
    results = hybrid_retrieval(question, k_each=2)
    comments = "\n".join([doc[:200] for doc, _ in results["comments"]])
    transcripts = "\n".join([doc[:200] for doc, _ in results["transcripts"]])
    summaries = "\n".join([doc[:300] for doc, _ in results["summaries"]])
    return f"COMMENTS:\n{comments}\n\nTRANSCRIPTS:\n{transcripts}\n\nSUMMARIES:\n{summaries}"


# ── TOOL 2: Sentiment Analysis ──
@tool
def sentiment_analysis(question: str) -> str:
    """Get sentiment data about a topic from YouTube comments.
    Use when asked how people feel or about emotional reactions."""
    df = pd.read_csv("../data/processed/comments_enriched.csv")
    docs, metas = semantic_retrieval(question, comments_collection, k=5)
    sentiments = [m["sentiment"] for m in metas]
    neg = sentiments.count("negative")
    neu = sentiments.count("neutral") 
    pos = sentiments.count("positive")
    overall = df["sentiment"].value_counts(normalize=True).round(3) * 100
    samples = "\n".join([doc[:150] for doc in docs[:3]])
    return f"Retrieved sentiment: {neg} negative, {neu} neutral, {pos} positive\nOverall dataset: negative={overall.get('negative',0):.1f}%, neutral={overall.get('neutral',0):.1f}%, positive={overall.get('positive',0):.1f}%\nSample comments:\n{samples}"


# ── TOOL 3: Topic Explorer ──
@tool  
def topic_explorer(question: str) -> str:
    """Find what topics and themes people discuss related to a question.
    Use when asked about main themes or discussion subjects."""
    docs, metas = semantic_retrieval(question, summaries_collection, k=3)
    result = "\n\n".join([f"Topic: {m.get('topic_name','unknown')}\n{doc[:300]}" 
                          for doc, m in zip(docs, metas)])
    return result


# ── TOOL 4: Entity Search ──
@tool
def entity_search(question: str) -> str:
    """Find comments mentioning specific companies, people, or AI tools.
    Use when asked about Google, Microsoft, OpenAI, Sam Altman, ChatGPT etc."""
    docs, metas = semantic_retrieval(question, comments_collection, k=5)
    results = []
    for doc, meta in zip(docs[:4], metas[:4]):
        entities = meta.get("entities", "[]")
        results.append(f"[{meta['sentiment']}] Entities: {entities}\n{doc[:200]}")
    return "\n\n".join(results)


# ── TOOL 5: Video Summarizer ──
@tool
def video_summarizer(question: str) -> str:
    """Get content from video transcripts about a specific topic.
    Use when asked about what a specific video says or covers."""
    docs, metas = semantic_retrieval(question, transcripts_collection, k=4)
    result = "\n\n".join([f"[{m.get('video_name','unknown')}]\n{doc[:300]}" 
                          for doc, m in zip(docs, metas)])
    return result


tools = [rag_qa, sentiment_analysis, topic_explorer, entity_search, video_summarizer]
print(f"Defined {len(tools)} tools:")
for t in tools:
    print(f"  - {t.name}: {t.description[:70]}...")

Defined 5 tools:
  - rag_qa: Answer general questions about AI and the future of work using 
    Yo...
  - sentiment_analysis: Get sentiment data about a topic from YouTube comments.
    Use when a...
  - topic_explorer: Find what topics and themes people discuss related to a question.
    ...
  - entity_search: Find comments mentioning specific companies, people, or AI tools.
    ...
  - video_summarizer: Get content from video transcripts about a specific topic.
    Use whe...


In [25]:
from langgraph.prebuilt import ToolNode, create_react_agent

# use llama-3.3-70b-versatile - confirmed working for tool calling on Groq
llm_agent = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0,
    max_tokens=512  # keep responses short to avoid token limit issues
)

# CRITICAL: explicit system prompt that prevents non-standard function call formats
# Groq rejects <function=name> format - must use standard JSON tool calls only
SYSTEM_PROMPT = """You are an AI analyst for the YouTube Intelligence Engine.
You have access to tools to answer questions about YouTube comments on AI and jobs.
IMPORTANT: When calling tools, use standard JSON format only.
Always call exactly one tool per response.
Available tools: rag_qa, sentiment_analysis, topic_explorer, entity_search, video_summarizer"""

from langchain_core.messages import SystemMessage

def agent_node(state: AgentState):
    # prepend system message to enforce standard tool call format
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_agent.bind_tools(
        tools,
        tool_choice="required",  # force tool use every time
        parallel_tool_calls=False  # one tool at a time
    ).invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)

def should_continue(state: AgentState):
    last = state["messages"][-1]
    # if last message has tool calls, execute them
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END

def final_answer_node(state: AgentState):
    # after tool runs, generate a clean final answer
    # limit history to avoid token issues - just system + question + tool result
    messages = [SystemMessage(content="""You are an AI analyst for the YouTube Intelligence Engine.
Based on the tool results provided, give a comprehensive, well-structured answer.
Ground your answer in the actual data from the tool results.""")]
    
    # include only the original question and tool result
    for msg in state["messages"]:
        if hasattr(msg, "content") and msg.content:
            messages.append(msg)
    
    response = llm_agent.invoke(messages)
    return {"messages": [response]}

builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_node("final", final_answer_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
builder.add_edge("tools", "final")
builder.add_edge("final", END)

graph = builder.compile()
print("Agent compiled with tool → final answer flow.")

Agent compiled with tool → final answer flow.


In [26]:
def ask_agent(question):
    """Send a question to the agent and return the final answer."""
    with mlflow.start_run(run_name=f"agent_{question[:30]}"):
        mlflow.log_param("question", question)
        
        try:
            result = graph.invoke({
                "messages": [HumanMessage(content=question)]
            })
            
            # get final answer
            final_answer = result["messages"][-1].content
            
            # log which tool was called
            for msg in result["messages"]:
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    tool_used = msg.tool_calls[0]["name"]
                    mlflow.log_param("tool_used", tool_used)
                    print(f"Tool selected: {tool_used}")
            
            mlflow.log_text(final_answer, "answer.txt")
            return final_answer
            
        except Exception as e:
            print(f"Agent error: {type(e).__name__} - falling back to RAG")
            results = hybrid_retrieval(question, k_each=3)
            comment_context = "\n\n".join([doc for doc, _ in results["comments"]])
            transcript_context = "\n\n".join([doc for doc, _ in results["transcripts"]])
            summary_context = "\n\n".join([doc for doc, _ in results["summaries"]])
            
            qa_prompt = ChatPromptTemplate.from_messages([
                ("system", "You are an AI analyst answering questions about YouTube comments on AI and jobs."),
                ("human", "Question: {question}\n\nComments: {comment_context}\n\nTranscripts: {transcript_context}\n\nSummaries: {summary_context}")
            ])
            chain = qa_prompt | llm
            response = chain.invoke({
                "question": question,
                "comment_context": comment_context,
                "transcript_context": transcript_context,
                "summary_context": summary_context
            })
            mlflow.log_param("tool_used", "fallback_rag")
            mlflow.log_text(response.content, "answer.txt")
            return response.content


# test all three questions
print("=" * 60)
print("Q1: How do people feel about AI taking their jobs?")
print("=" * 60)
answer1 = ask_agent("How do people feel about AI taking their jobs?")
print(answer1[:500])

print("\n" + "=" * 60)
print("Q2: What does Google do in the AI space according to comments?")
print("=" * 60)
answer2 = ask_agent("What does Google do in the AI space according to comments?")
print(answer2[:500])

print("\n" + "=" * 60)
print("Q3: What are the main topics people discuss about AI and work?")
print("=" * 60)
answer3 = ask_agent("What are the main topics people discuss about AI and work?")
print(answer3[:500])

Q1: How do people feel about AI taking their jobs?
Tool selected: sentiment_analysis
Based on the tool results, it appears that people have mixed feelings about AI taking their jobs. The sentiment analysis shows that 50.2% of the overall dataset has a negative sentiment, indicating that many people are concerned or upset about the potential job loss due to AI. 

On the other hand, 18.7% of the dataset has a positive sentiment, suggesting that some individuals see AI as an opportunity or are not worried about job loss. The remaining 31.2% have a neutral sentiment, which may indic

Q2: What does Google do in the AI space according to comments?
Tool selected: entity_search
Based on the comments provided, it appears that Google is involved in the AI space in several ways. Here are some key points that can be gathered from the comments:

1. **AI-generated comments**: Many comments are AI-generated, which suggests that Google is using AI to generate content, possibly for testing or demonstra

In [1]:
import json
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), '.env'))
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY, temperature=0)

c:\Users\amirh\Documents\projects\youtube-intelligence-engine\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── QUERY ANALYSIS ──
# explicitly demonstrates query classification before routing
# this is a named deliverable in the project brief
# shows the system understands WHAT is being asked before deciding HOW to retrieve

QUERY_ANALYSIS_PROMPT = """Analyze this user query and classify it.

Query: {query}

Respond in this exact JSON format only:
{{
    "intent": "one of: factual_qa, sentiment_inquiry, topic_exploration, entity_search, summarization",
    "entities_mentioned": ["list any companies, people, or AI tools mentioned"],
    "sentiment_requested": "one of: any, positive, negative, neutral",
    "complexity": "one of: simple, complex",
    "reasoning": "one sentence explanation"
}}"""

def analyze_query(query):
    """
    Classifies user query before routing to appropriate tool.
    Identifies intent, entities, sentiment preference, and complexity.
    This drives intelligent tool selection in the agent.
    """
    response = llm.invoke(QUERY_ANALYSIS_PROMPT.format(query=query))
    try:
        text = response.content.strip()
        start = text.find("{")
        end = text.rfind("}") + 1
        return json.loads(text[start:end])
    except:
        return {"intent": "factual_qa", "entities_mentioned": [], 
                "sentiment_requested": "any", "complexity": "simple",
                "reasoning": "parse error - defaulting to factual QA"}

# test query analysis on different query types
test_queries = [
    "What do people think about AI replacing software developers?",
    "What does Sam Altman say about OpenAI?",
    "Summarize the main topics people discuss about AI and jobs",
    "What are the most negative views about automation?",
    "How do artists feel about AI-generated content?"
]

print("=== QUERY ANALYSIS DEMONSTRATION ===\n")
for query in test_queries:
    analysis = analyze_query(query)
    print(f"Query: {query[:60]}")
    print(f"  Intent:    {analysis.get('intent')}")
    print(f"  Entities:  {analysis.get('entities_mentioned')}")
    print(f"  Sentiment: {analysis.get('sentiment_requested')}")
    print(f"  Complexity:{analysis.get('complexity')}")
    print(f"  Reasoning: {analysis.get('reasoning')}")
    print()

=== QUERY ANALYSIS DEMONSTRATION ===

Query: What do people think about AI replacing software developers?
  Intent:    sentiment_inquiry
  Entities:  []
  Sentiment: any
  Complexity:simple
  Reasoning: The user is asking for opinions or thoughts on a specific topic, indicating a desire to understand the sentiment or attitudes of people towards AI replacing software developers.

Query: What does Sam Altman say about OpenAI?
  Intent:    factual_qa
  Entities:  ['Sam Altman', 'OpenAI']
  Sentiment: any
  Complexity:simple
  Reasoning: The user is seeking information about Sam Altman's statements or opinions regarding OpenAI, indicating a straightforward factual inquiry.

Query: Summarize the main topics people discuss about AI and jobs
  Intent:    topic_exploration
  Entities:  []
  Sentiment: any
  Complexity:simple
  Reasoning: The user is asking for a general overview of the main topics related to AI and jobs, indicating a desire to explore the subject without specifying particular 

In [3]:
# ── QUERY ANALYSIS → TOOL ROUTING ──
# shows how query analysis informs which tool gets selected
# connects the two concepts: understand query first, then route appropriately

print("=== QUERY ANALYSIS → TOOL ROUTING ===\n")

routing_map = {
    "sentiment_inquiry":  "sentiment_analysis",
    "topic_exploration":  "topic_explorer", 
    "entity_search":      "entity_search",
    "summarization":      "video_summarizer",
    "factual_qa":         "rag_qa"
}

for query in test_queries:
    analysis = analyze_query(query)
    recommended_tool = routing_map.get(analysis.get("intent"), "rag_qa")
    print(f"Query:          {query[:55]}")
    print(f"Detected intent:{analysis.get('intent')}")
    print(f"Routed to tool: {recommended_tool}")
    print()

=== QUERY ANALYSIS → TOOL ROUTING ===

Query:          What do people think about AI replacing software develo
Detected intent:sentiment_inquiry
Routed to tool: sentiment_analysis

Query:          What does Sam Altman say about OpenAI?
Detected intent:factual_qa
Routed to tool: rag_qa

Query:          Summarize the main topics people discuss about AI and j
Detected intent:topic_exploration
Routed to tool: topic_explorer

Query:          What are the most negative views about automation?
Detected intent:sentiment_inquiry
Routed to tool: sentiment_analysis

Query:          How do artists feel about AI-generated content?
Detected intent:sentiment_inquiry
Routed to tool: sentiment_analysis

